# 04 Validation And Comparison

This notebook compares the custom RK4 solver against the SciPy baseline on the same Lorenz-1960 benchmark.

## Validation goals

- Compare trajectories visually
- Compare each state variable over time
- Quantify the discrepancy using error metrics
- Check whether the fixed-step RK4 result improves when the step size is reduced
- Judge whether the numerical baseline is reliable enough to support later ANN experiments


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from lorenz1960_baseline import (
    DEFAULT_CONFIG,
    compute_error_metrics,
    lorenz1960_rhs,
    make_uniform_grid,
    plot_3d_trajectory,
    plot_error_curves,
    plot_state_time_series,
    rk4_solve,
    solve_lorenz1960_scipy,
    subsample_uniform_solution,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:.12e}")


## Run both solvers on the shared comparison grid

In [ ]:
config = DEFAULT_CONFIG
comparison_grid = make_uniform_grid(config.t_span, step=config.rk4_step)
scipy_method = config.scipy_method

ts_rk4, ys_rk4 = rk4_solve(
    lorenz1960_rhs,
    config.t_span,
    config.initial_state,
    config.rk4_step,
)
ts_scipy, ys_scipy, scipy_solution = solve_lorenz1960_scipy(config=config, t_eval=comparison_grid)

assert len(ts_rk4) == len(ts_scipy) == len(comparison_grid)


## Quantitative error metrics

The SciPy solution is used here as the numerical reference because it uses tighter adaptive error control. This does **not** make it analytical truth, but it is a reasonable benchmark for checking whether the custom RK4 implementation is behaving consistently.

In [ ]:
metrics_primary = compute_error_metrics(reference=ys_scipy, candidate=ys_rk4)
metrics_primary


## Final-state difference

In [ ]:
final_state_comparison = pd.DataFrame(
    {
        "state": ["x", "y", "z"],
        "rk4_final": ys_rk4[-1],
        "scipy_final": ys_scipy[-1],
        "absolute_difference": abs(ys_rk4[-1] - ys_scipy[-1]),
    }
)
final_state_comparison


## Time-series overlay

In [ ]:
plot_state_time_series(
    ts_rk4,
    ys_rk4,
    title="RK4 and SciPy trajectories on the same grid",
    overlay=ys_scipy,
    overlay_name=f"SciPy {scipy_method}",
)
plt.show()


## 3D trajectory overlay

In [ ]:
plot_3d_trajectory(
    ys_rk4,
    title="3D trajectory comparison: RK4 vs SciPy",
    overlay=ys_scipy,
)
plt.show()


## Absolute error curves

In [ ]:
plot_error_curves(
    ts_rk4,
    reference=ys_scipy,
    candidate=ys_rk4,
    title="Absolute error of RK4 against the SciPy baseline",
)
plt.show()


## RK4 step-halving check

If the RK4 implementation is working correctly, reducing the fixed step size should reduce the discrepancy against the SciPy reference.

In [ ]:
ts_rk4_fine, ys_rk4_fine = rk4_solve(
    lorenz1960_rhs,
    config.t_span,
    config.initial_state,
    config.rk4_step / 2.0,
)
if len(ts_rk4_fine[::2]) == len(ts_rk4) and (abs(ts_rk4_fine[::2] - ts_rk4) < 1e-15).all():
    ys_rk4_fine_on_primary_grid = ys_rk4_fine[::2]
else:
    ys_rk4_fine_on_primary_grid = subsample_uniform_solution(ts_rk4_fine, ys_rk4_fine, ts_rk4)

metrics_fine = compute_error_metrics(reference=ys_scipy, candidate=ys_rk4_fine_on_primary_grid)
metrics_fine


## Reliability discussion

In [ ]:
primary_rmse = float(metrics_primary.loc[metrics_primary["state"] == "combined_l2", "rmse"].iloc[0])
fine_rmse = float(metrics_fine.loc[metrics_fine["state"] == "combined_l2", "rmse"].iloc[0])
relative_gap = abs(fine_rmse - primary_rmse) / max(primary_rmse, fine_rmse, 1e-30)

if fine_rmse < primary_rmse * (1.0 - 1e-3):
    verdict = "Reducing the RK4 step size decreases the discrepancy, which supports the expected convergence behavior."
elif relative_gap <= 1e-3:
    verdict = "The step-halving check is essentially unchanged at this scale, which suggests the RK4-vs-SciPy discrepancy has already reached a numerical floor for this comparison."
else:
    verdict = "The step-halving check changed in an unexpected way, so the baseline should be investigated further before using it for ANN targets."

print("Primary combined RMSE:", primary_rmse)
print("Fine-step combined RMSE:", fine_rmse)
print("Relative gap:", relative_gap)
print("Reliability note:", verdict)


## Interpretation

Use the metrics and plots above for the baseline decision:

- If RK4 and SciPy agree closely on `[0, 1]`, the pipeline is trustworthy enough to generate ANN training targets for this benchmark.
- If the discrepancy is large or does not shrink under step halving, the RK4 implementation or the numerical settings should be revised before any ANN experiment begins.
- The SciPy solution is still a numerical approximation, so future work can tighten tolerances further or compare against additional reference settings if a supervisor asks for stronger validation.